# 😊 Face Emotion Detection — Model Training
### Run this notebook in Google Colab with GPU enabled

## Step 1 — Enable GPU
> **Runtime → Change runtime type → Hardware accelerator → GPU → Save**

In [ ]:
# Step 2 — Upload your Kaggle API key
from google.colab import files
files.upload()  # Upload kaggle.json here

In [ ]:
# Step 3 — Setup Kaggle
import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.system('cp kaggle.json /root/.kaggle/')
os.system('chmod 600 /root/.kaggle/kaggle.json')
print('Kaggle API configured!')

In [ ]:
# Step 4 — Download FER-2013 Dataset
os.system('kaggle datasets download -d msambare/fer2013')
os.system('unzip -q fer2013.zip -d fer2013')
print('Dataset downloaded and extracted!')

In [ ]:
# Step 5 — Install & Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

print('TensorFlow version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Step 6 — Check Dataset Structure
import os

train_dir = 'fer2013/train'
test_dir  = 'fer2013/test'

emotions = os.listdir(train_dir)
print('Emotion classes:', emotions)

for emotion in emotions:
    count = len(os.listdir(os.path.join(train_dir, emotion)))
    print(f'  {emotion}: {count} images')

In [ ]:
# Step 7 — Data Augmentation & Generators
IMG_SIZE  = 48
BATCH     = 64
EPOCHS    = 50

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH,
    class_mode='categorical',
    shuffle=True
)

test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH,
    class_mode='categorical',
    shuffle=False
)

print('Class indices:', train_gen.class_indices)

In [ ]:
# Step 8 — Build CNN Model
model = Sequential([
    # Block 1
    Conv2D(64, (3,3), activation='relu', padding='same', input_shape=(48, 48, 1)),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    # Block 2
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    # Block 3
    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    # Fully Connected
    Flatten(),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(7, activation='softmax')  # 7 emotions
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Step 9 — Train the Model
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
    ModelCheckpoint('best_emotion_model.h5', monitor='val_accuracy', save_best_only=True)
]

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=test_gen,
    callbacks=callbacks
)

In [ ]:
# Step 10 — Plot Training Results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

# Final accuracy
loss, acc = model.evaluate(test_gen)
print(f'\nFinal Test Accuracy: {acc*100:.2f}%')

In [ ]:
# Step 11 — Save & Download the Model
model.save('emotion_model.h5')
print('Model saved as emotion_model.h5')

# Download to your PC
from google.colab import files
files.download('emotion_model.h5')
print('Download started! Save it in your Streamlit app folder.')

## ✅ Done!
- `emotion_model.h5` is downloaded to your PC
- Place it in the same folder as `app.py`
- Run `streamlit run app.py`